# Content Clustering Evaluation

This notebook groups photos by **semantic content** using K-Means on deep
embeddings. It helps choose the optimal number of clusters *K* via the
**elbow method** (inertia) and **silhouette score**, then runs the final
clustering and visualises the results.

## Prerequisites

- Embeddings are read from the parquet cache produced by notebook 02, **or**
  computed fresh from a folder of images.
- Set `IMAGE_DIR` below to point at the photo collection you want to cluster.

## Outputs

| File | Description |
|------|-------------|
| `results/content_cluster_labels.parquet` | Per-image cluster assignments |
| `results/content_cluster_viz.png` | 2-D UMAP / t-SNE scatter plot |
| `results/silhouette_vs_k.png` | Silhouette score vs. K curve |

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("content_cluster_eval")

IMAGE_DIR     = PROJECT_ROOT / "data" / "sample"
RESULTS_DIR   = PROJECT_ROOT / "results"
CACHE_PARQUET = RESULTS_DIR / "dedup_embeddings.parquet"   # reuse embeddings from nb-02 if present
VIZ_PATH      = RESULTS_DIR / "content_cluster_viz.png"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
log.info("Paths configured. PROJECT_ROOT=%s", PROJECT_ROOT)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from src.content_cluster import load_or_compute_embeddings

# Load or compute embedding matrix
embeddings_df = load_or_compute_embeddings(
    cache_path=str(CACHE_PARQUET),
    image_dir=str(IMAGE_DIR),
    model_name="efficientnet_b0",
    img_size=(224, 224),
    batch_size=64,
)

# Extract the raw embedding matrix (columns starting with 'emb_')
emb_cols = [c for c in embeddings_df.columns if c.startswith("emb_")]
X = embeddings_df[emb_cols].values
log.info("Embedding matrix shape: %s", X.shape)

# ── Elbow + silhouette loop ──────────────────────────────────────────────────
K_RANGE = range(2, 21)
inertias, silhouettes = [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil = silhouette_score(X, labels, sample_size=min(2000, len(X)), random_state=42)
    silhouettes.append(sil)
    log.info("K=%2d  inertia=%.1f  silhouette=%.4f", k, km.inertia_, sil)

# ── Plot silhouette scores vs. K ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_RANGE), inertias, marker="o", color="steelblue")
axes[0].set_xlabel("Number of Clusters K")
axes[0].set_ylabel("Inertia (within-cluster SSE)")
axes[0].set_title("Elbow Method")

axes[1].plot(list(K_RANGE), silhouettes, marker="s", color="darkorange")
axes[1].set_xlabel("Number of Clusters K")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score vs. K")
best_k = list(K_RANGE)[int(np.argmax(silhouettes))]
axes[1].axvline(x=best_k, color="crimson", linestyle="--", label=f"Best K={best_k}")
axes[1].legend()

plt.tight_layout()
sil_fig_path = RESULTS_DIR / "silhouette_vs_k.png"
plt.savefig(str(sil_fig_path), dpi=150)
plt.show()
log.info("Best K by silhouette: %d — figure saved to %s", best_k, sil_fig_path)

In [ ]:
from src.content_cluster import run_content_clustering, visualise_clusters

# ── Run final K-Means with the optimal K ─────────────────────────────────────
FINAL_K = best_k   # override here if you prefer a different value
log.info("Running content clustering with K=%d …", FINAL_K)

cluster_df = run_content_clustering(
    embeddings_df=embeddings_df,
    n_clusters=FINAL_K,
    random_state=42,
)

out_parquet = RESULTS_DIR / "content_cluster_labels.parquet"
cluster_df.to_parquet(str(out_parquet), index=False)
log.info("Cluster labels saved to %s", out_parquet)

# ── Build visualisation (UMAP / t-SNE scatter) ────────────────────────────────
log.info("Generating cluster visualisation (this may take a moment) …")
visualise_clusters(
    cluster_df=cluster_df,
    embeddings=X,
    n_clusters=FINAL_K,
    method="umap",          # "umap" or "tsne"
    save_path=str(VIZ_PATH),
)
log.info("Visualisation saved to %s", VIZ_PATH)

In [ ]:
# ── Display the cluster visualisation inline ──────────────────────────────────
if VIZ_PATH.exists():
    print(f"Cluster visualisation ({FINAL_K} clusters):")
    display(IPImage(filename=str(VIZ_PATH)))
else:
    log.warning("Visualisation file not found at %s", VIZ_PATH)
    print("[WARNING] Visualisation file was not created — check logs above.")

# ── Summary statistics ───────────────────────────────────────────────────────
print("\nCluster size distribution:")
print(
    cluster_df["Cluster"]
    .value_counts()
    .rename_axis("Cluster")
    .reset_index(name="Count")
    .to_string(index=False)
)